# Tool Result Caching

The OpenAI Agents SDK routes tool execution through the `Runner`, not through
`Model.get_response`, so tool caching cannot be implemented as middleware
over the model interface. `cached_tool` is a decorator that wraps a Python
callable *before* it becomes a `@function_tool`, memoising its result in
Redis by a hash of the arguments.

Good candidates for `cached_tool`:

- API calls that are deterministic or cheaply stale-tolerant (weather,
  exchange rates, company lookups).
- Expensive calculations (embedding a long document, statistical aggregates).
- Database reads that don't need to reflect live state.

Avoid for:

- Tools that write, delete, or otherwise have side effects.
- Tools whose output must reflect real-time state.


## Setup


In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY not found. Please set it in .env or environment.")

REDIS_URL = os.environ.get("REDIS_URL", "redis://localhost:6379")
print(f"Redis URL: {REDIS_URL}")


In [ ]:
from agents import Agent, Runner, function_tool
from redis_openai_agents import cached_tool


## Simulate an expensive tool

Pretend `lookup_company` hits a paid API with 500 ms latency. The
`cached_tool` wrapper memoises the call in Redis; the second call with the
same argument is instant.


In [ ]:
import time

call_count = {"n": 0}


@cached_tool(
    name="lookup_company",
    redis_url=REDIS_URL,
    ttl=3600,  # 1 hour
)
def lookup_company(ticker: str) -> str:
    call_count["n"] += 1
    time.sleep(0.5)  # simulated API latency
    return f"{ticker.upper()}: Consumer Tech, Large Cap"


# First call: cache miss
start = time.time()
result1 = lookup_company("aapl")
elapsed1 = (time.time() - start) * 1000
print(f"1st call: {result1!r} ({elapsed1:.0f} ms, calls={call_count['n']})")

# Second call: cache hit
start = time.time()
result2 = lookup_company("aapl")
elapsed2 = (time.time() - start) * 1000
print(f"2nd call: {result2!r} ({elapsed2:.1f} ms, calls={call_count['n']})")
print(f"Speedup: {elapsed1 / elapsed2:.0f}x")


## Volatile arguments bypass the cache

Arguments whose name matches a volatile pattern (`timestamp`, `now`, `date`,
etc.) always prevent caching. The underlying function runs every time.


In [ ]:
count = {"n": 0}


@cached_tool(name="forecast", redis_url=REDIS_URL)
def forecast(city: str, timestamp: float) -> str:
    count["n"] += 1
    return f"forecast for {city} at t={timestamp:.0f}"


forecast("Paris", time.time())
forecast("Paris", time.time())
print(f"Calls made: {count['n']} (both calls ran because of `timestamp`)")


## Ignoring non-deterministic arguments

Some arguments (like trace IDs) change every call but don't affect the
result. Declare them as ignored and they get stripped from the cache key.


In [ ]:
count = {"n": 0}


@cached_tool(
    name="hash_text",
    redis_url=REDIS_URL,
    ignored_arg_names={"trace_id"},
)
def hash_text(text: str, trace_id: str) -> int:
    count["n"] += 1
    return hash(text)


hash_text("hello", trace_id="t-1")
hash_text("hello", trace_id="t-2")
hash_text("hello", trace_id="t-3")
print(f"Calls: {count['n']} (only the first hit the function)")


## Side-effecting tool names are never cached

Tools whose name starts with a side-effect prefix (`send_`, `delete_`,
`create_`, `update_`, ...) are automatically excluded from caching even if
their arguments match.


In [ ]:
count = {"n": 0}


@cached_tool(name="send_email", redis_url=REDIS_URL)
def send_email(to: str, body: str) -> str:
    count["n"] += 1
    return "ok"


send_email("a@b.com", "hi")
send_email("a@b.com", "hi")
print(f"Emails sent: {count['n']} (name starts with send_, so not cached)")


## Using with an Agent

Stack `@cached_tool` *inside* `@function_tool` so the cache applies to the
underlying callable before the SDK exposes it to the model.


In [ ]:
@function_tool
@cached_tool(name="get_quote", redis_url=REDIS_URL, ttl=3600)
def get_quote(ticker: str) -> str:
    """Return the last close price for a stock ticker."""
    # Simulated - would be a real API call in production
    return f"{ticker}: $175.43"


agent = Agent(
    name="finance-bot",
    instructions="You answer questions about stocks. Use get_quote for prices.",
    tools=[get_quote],
)

# First run: agent calls get_quote, result is cached
result1 = await Runner.run(agent, "What is the price of AAPL?")
print(f"1st run: {result1.final_output}")

# Second run on same ticker: get_quote returns from cache
result2 = await Runner.run(agent, "And again, what is AAPL at?")
print(f"2nd run: {result2.final_output}")


## Summary

- `cached_tool` memoises a callable's result in Redis by a hash of its
  arguments.
- Apply `@cached_tool(...)` *inside* `@function_tool(...)` so the agent's
  tool interface stays unchanged.
- `volatile_arg_names` (default: `timestamp`, `now`, ...) bypass caching
  entirely.
- `ignored_arg_names` are stripped from the cache key so they don't bust
  cache hits.
- `side_effect_prefixes` (default: `send_`, `delete_`, `create_`, ...)
  disable caching based on the tool name.
- Works for both sync and async callables.
